# Downloading Data from the Abacus Dataverse

I will be retrieving monthly Canadian Labour Force Survey data public files from the Abacus Dataverse hosted by the University of British Columbia library (https://abacus.library.ubc.ca/).
While the usual way to access data from the repository is graphically using its point-and-click web interface, all Dataverses are also able to serve their files through API (https://guides.dataverse.org/en/5.6/).

My intention is to eventually download the entire available history of LFS data to do some example analysis.
Hence it is neither economical nor easily replicable to use the point-and-click interface.
This notebook will be a demonstration of how to get metadata and request (publicly-available) files from the Dataverse server.

In [1]:
import requests
import json
import re
import time
import pandas as pd
import pymongo
import datetime

We set up some global variables here to make life easy later on.

In [2]:
endpoints = {
    'search': 'https://abacus.library.ubc.ca/api/search',
    'metadata': 'https://abacus.library.ubc.ca/api/datasets/:persistentId',
    'file_download': 'https://abacus.library.ubc.ca/api/access/datafile/:persistentId'
}

# 1. Search for Datasets

First, we need to retrieve metadata of relevant datasets to fetch later.

In [3]:
search_params = {
    'q': 'title:Labour Force Survey',
    'type': 'dataset',
    'per_page': 100
}
response = requests.get(url=endpoints['search'], params=search_params)
print('Response Status:', response.status_code)

Response Status: 200


We know from the documentation that there is per-page rate limiting, so we will recover the full list of relevant results through a simple loop that accounts for the possibility of failure due to accidentally going over the rate limit or otherwise.

In [4]:
items = response.json()['data']['items'].copy()
for start in range(search_params['per_page'], response.json()['data']['total_count']+1, search_params['per_page']):
    search_params['start'] = start
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['search'], params=search_params)
        if response.status_code == 200:
            items.extend(response.json()['data']['items'])
            fail_response = False
            print('Success:', start)
            time.sleep(.1)
        else:
            print('Fail:', start)
            time.sleep(5)
print('Total Returned Items:', len(items))

Success: 100
Success: 200
Success: 300
Success: 400
Success: 500
Success: 600
Success: 700
Success: 800
Total Returned Items: 892


We then use regex to recover the appropriate files.
Canada uses both American and British spelling and the Labour Force Survey is sometimes also abbreviated as the LFS, so regex is appropriate here.

In [5]:
lfs_re = re.compile(r'(?i:labou?r force survey|lfs)')
lfs_items = list(filter(lambda i: lfs_re.match(i['name']), items))
lfs_items.sort(key=lambda i: i['name'], reverse=True)
print('Labour Force Survey datasets:', len(lfs_items))

Labour Force Survey datasets: 62


Before writing the data, do some simple transformations that will save a lot of time and work later.
First is to convert datetime strings to datetime format.

In [6]:
for i in range(len(lfs_items)):
    for f in ['published_at', 'createdAt', 'updatedAt']:
        lfs_items[i][f] = datetime.datetime.strptime(lfs_items[i][f], r'%Y-%m-%dT%H:%M:%SZ')

Second is to recover the LFS series number (i.e. the Year number).
This one is important because previous surveys in the program have been updated and republished since their original publication.
We generally want the latest versions because StatCan has fixed data issues and incorrect survey weights.

In [7]:
yr_re = re.compile(r'\,\s*(\d{4})\W*$')
for i in range(len(lfs_items)):
    lfs_items[i]['seriesNum'] = int(yr_re.search(lfs_items[i]['name']).group(1))

I need a reference file so that I do not keep disturbing the API unnecessarily later.
After visually inspecting the list briefly to be sure that this is what I want, I will export the saved results to a MongoDB database.

In [8]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_col_datasets = lfs_db['datasets']

Pinged your deployment. You successfully connected to MongoDB!


Now we can upload the metadata using the Python MongoDB Driver API.

In [9]:
new_ids = lfs_col_datasets.insert_many(lfs_items).inserted_ids
print(len(new_ids))
print(new_ids[:5])

62
[ObjectId('6470cfd3661078467fabae4f'), ObjectId('6470cfd3661078467fabae50'), ObjectId('6470cfd3661078467fabae51'), ObjectId('6470cfd3661078467fabae52'), ObjectId('6470cfd3661078467fabae53')]


The Driver makes it quite convenient now to retrieve the specific fields we will need going forward.
I also make use of its aggregator pipeline functionality to recover only the versions of interest of republished LFS's.
Below, briefly, I use the `$group` aggregator function to get the unique dataset for each year (`seriesNum`) that has the highest `versionId`.

In [10]:
pids = list(lfs_col_datasets.aggregate([
    {'$group': {
        '_id': '$seriesNum',
        'name': {
            '$top': {
                'output': '$name',
                'sortBy': {'versionId': -1}
            }
        },
        'global_id': {
            '$top': {
                'output': '$global_id',
                'sortBy': {'versionId': -1}
            }
        }
    }},
    {'$sort': {
        '_id': -1
    }}
]))

Things are done.
We can close the connection to the database.

In [11]:
client.close()

# 2. Get Dataset Metadata

We now call a second API to recover the metadata and, especially, the file list of each dataset.

In [12]:
search_params = {
    'persistentId': pids[0]['global_id']
}
response = requests.get(url=endpoints['metadata'], params=search_params)
print('Response Status:', response.status_code)

Response Status: 200


The metadata of interest are stored in the `files` subsection.

In [13]:
items = response.json()['data']['latestVersion']['files']

We are going to loop through the remaining data sets and compile the datafiles list.

In [14]:
for ds in pids[1:]:
    search_params['persistentId'] = ds['global_id']
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['metadata'], params=search_params)
        if response.status_code == 200:
            items.extend(response.json()['data']['latestVersion']['files'])
            fail_response = False
            print('Success:', ds)
            time.sleep(.1)
        else:
            print('Fail:', ds)
            time.sleep(5)
print('Total Returned Items:', len(items))

Success: {'_id': 2022, 'name': 'Labour Force Survey, 2022', 'global_id': 'hdl:11272.1/AB2/SRAU7E'}
Success: {'_id': 2021, 'name': 'Labour Force Survey, 2021', 'global_id': 'hdl:11272.1/AB2/HP9TEK'}
Success: {'_id': 2020, 'name': 'Labour Force Survey, 2020', 'global_id': 'hdl:11272.1/AB2/GGXMM2'}
Success: {'_id': 2019, 'name': 'Labour Force Survey, 2019', 'global_id': 'hdl:11272.1/AB2/ATGWRX'}
Success: {'_id': 2018, 'name': 'Labour Force Survey, 2018', 'global_id': 'hdl:11272.1/AB2/CFRSC0'}
Success: {'_id': 2017, 'name': 'Labour Force Survey, 2017', 'global_id': 'hdl:11272.1/AB2/PO8CWI'}
Success: {'_id': 2016, 'name': 'Labour Force Survey, 2016', 'global_id': 'hdl:11272.1/AB2/SRHMXU'}
Success: {'_id': 2015, 'name': 'Labour Force Survey, 2015', 'global_id': 'hdl:11272.1/AB2/N1N5S6'}
Success: {'_id': 2014, 'name': 'Labour Force Survey, 2014', 'global_id': 'hdl:11272.1/AB2/B9RMKT'}
Success: {'_id': 2013, 'name': 'Labour Force Survey, 2013', 'global_id': 'hdl:11272.1/AB2/1CUHBA'}
Success: {

As before, we will save the data files metadata into the MongoDB database for quick retrieval in the future.

In [15]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_col_datafiles = lfs_db['datafiles']

Pinged your deployment. You successfully connected to MongoDB!


In [16]:
new_ids = lfs_col_datafiles.insert_many(items).inserted_ids
print(len(new_ids))
print(new_ids[:5])

782
[ObjectId('6470d028661078467fabae8e'), ObjectId('6470d028661078467fabae8f'), ObjectId('6470d028661078467fabae90'), ObjectId('6470d028661078467fabae91'), ObjectId('6470d028661078467fabae92')]


Close the client once done.

In [17]:
client.close()